# Coffee LoRA, option B: self-curated style LoRA (Wan2.1-1.3B, free T4)
Pretrained Wan generates candidate coffee clips -> automatic filter -> LoRA trained on the kept clips -> base vs LoRA comparison -> ad with the LoRA. Runtime > T4 GPU.
**Honest framing:** the LoRA learns from Wan's own outputs, so it can only make the style more consistent, not add new knowledge. Evaluate it on held-out prompts.

**1. Upload `video_gen_upload.zip`** (re-zip the repo AFTER adding the new scripts), then set up.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import glob, os
!mkdir -p /content/repo && unzip -oq /content/video_gen_upload.zip -d /content/repo
WORK = os.path.dirname(glob.glob("/content/repo/**/train_lora.py", recursive=True)[0])
%cd {WORK}
!apt-get -q install -y fonts-noto-core > /dev/null
!pip -q install -r requirements.txt
!pip -q uninstall -y torchao
!pip -q install uroman
!ls coffee_lora

**2. Generate 40 candidate clips** (about 30 min). Safe to rerun after a disconnect: existing clips are skipped.

In [ ]:
!python coffee_lora/make_synth_clips.py --n 40 --out data/coffee_synth

**3. Filter automatically** (CPU, seconds), then back up the dataset.

In [ ]:
!python coffee_lora/curate_clips.py --data data/coffee_synth --keep 30
!head -4 data/coffee_synth/curation.csv
!zip -rq /content/coffee_synth_data.zip data/coffee_synth
from google.colab import files
files.download("/content/coffee_synth_data.zip")

In [ ]:
import json
from IPython.display import Video, display
kept = [json.loads(l) for l in open("data/coffee_synth/metadata.jsonl", encoding="utf-8")]
print(len(kept), "kept clips; glance at 3:")
for m in kept[:3]:
    print(m["caption"]); display(Video("data/coffee_synth/" + m["file"], embed=True, width=216))

**4. Smoke test** (memory check, includes the one-time cache build). Send the peak GPU number.

In [ ]:
!python train_lora.py --data_dir data/coffee_synth --steps 10 --accum 1 --height 512 --width 288 --frames 17 --out outputs/coffee_selfdata

**5. Train the LoRA** (rank 16, 300 steps; reuses the cache).

In [ ]:
!python train_lora.py --data_dir data/coffee_synth --steps 300 --accum 2 --rank 16 --height 512 --width 288 --frames 17 --out outputs/coffee_selfdata

**6. Compare base vs LoRA** with the same seeds: the 4 ad-scene prompts plus 2 HELD-OUT prompts that were never used for the training clips.

In [ ]:
import sys, shlex
sys.path.insert(0, "coffee_ad")
from gen_scenes import PROMPTS
PREFIX = "Ethiopian coffee commercial style: "
HELD_OUT = ["Close-up of roasted coffee beans in a woven basket, soft window light, shallow depth of field, cinematic",
            "Small white coffee cups on a tray with rising steam, dark background, golden highlights, slow camera push-in"]
args = " ".join("--prompt " + shlex.quote(PREFIX + p) for p in PROMPTS + HELD_OUT)
!python infer.py --lora_dir outputs/coffee_selfdata --out compare_selfdata {args} --height 512 --width 288 --frames 33

In [ ]:
from IPython.display import Video, display
for k in range(6):
    for tag in ["base", "lora"]:
        print(tag, "prompt", k + 1, "(held-out)" if k >= 4 else "(ad scene)")
        display(Video(f"compare_selfdata/{tag}_{k}.mp4", embed=True, width=216))
!python coffee_lora/curate_clips.py --report compare_selfdata

**7. Only if the LoRA looks better: make the ad with it.**

In [ ]:
!python coffee_ad/gen_scenes.py --out scenes_lora --lora_dir outputs/coffee_selfdata --prefix "{PREFIX}"
!python coffee_ad/assemble.py --scenes scenes_lora --out ad_lora_silent.mp4
!python coffee_ad/voiceover.py --video ad_lora_silent.mp4 --out ethiopian_coffee_ad_lora.mp4

**8. Download everything** before the session ends.

In [ ]:
!zip -rq /content/selfdata_results.zip compare_selfdata outputs/coffee_selfdata scenes_lora ad_lora_silent.mp4 ethiopian_coffee_ad_lora.mp4 voice.wav data/coffee_synth/curation.csv -x "*cache*"
from google.colab import files
files.download("/content/selfdata_results.zip")